# Harvest-Season Rain → Production Analysis

**Thesis:** Rain during the harvest window (August–October) is the primary precipitation damage channel — mold, nut rot, and mechanical harvesting complications.

**Data:** ERA5 monthly `tp` (total precipitation) rebuilt from `data_stream-moda_stepType-avgad.nc`, production-weighted across 6 Black Sea provinces (Ordu 31%, Giresun 16%, Samsun 14%, Sakarya 13%, Düzce 11%, Trabzon 6%). Units: mm/month.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})

## 1. Load & Merge

In [ ]:
precip = pd.read_csv("../data/raw/era5_precip_monthly.csv")
prod   = pd.read_csv("../data/raw/faostat/turkey_hazelnut_production.csv")

df = precip.merge(prod, on="year", how="inner").sort_values("year").reset_index(drop=True)
df["prod_chg"] = df["production_mt"].pct_change()   # YoY fractional change
df = df.dropna(subset=["prod_chg"]).copy()

print(f"Panel: {df.year.min()}–{df.year.max()}, n={len(df)}")
print(f"\nHarvest-window columns (mm/month):")
print(df[["aug_mm", "sep_mm", "oct_mm", "harvest_mm"]].describe().round(1))

## 2. Harvest-Window Distributions

In [ ]:
cols_labels = [
    ("aug_mm",     "August (mm)"),
    ("sep_mm",     "September (mm)"),
    ("oct_mm",     "October (mm)"),
    ("harvest_mm", "Aug–Oct total (mm)"),
]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, (col, label) in zip(axes, cols_labels):
    ax.hist(df[col], bins=14, edgecolor="white", color="steelblue", alpha=0.85)
    ax.axvline(df[col].mean(),   color="tomato", lw=1.5, ls="--", label=f"μ={df[col].mean():.0f}")
    ax.axvline(df[col].median(), color="orange", lw=1.5, ls=":",  label=f"med={df[col].median():.0f}")
    ax.set_title(label, fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle("Harvest-Window Precipitation Distributions", y=1.02)
plt.tight_layout()
plt.show()

## 3. Correlations with YoY Production Change

In [ ]:
print(f"{'Column':<22} {'Pearson r':>10} {'p':>8}   {'Spearman ρ':>12} {'p':>8}")
print("-" * 68)
for col, label in cols_labels:
    r,  p  = stats.pearsonr(df[col], df["prod_chg"])
    sr, sp = stats.spearmanr(df[col], df["prod_chg"])
    sig = " *" if min(p, sp) < 0.10 else ""
    print(f"{label:<22} {r:>+10.3f} {p:>8.3f}   {sr:>+12.3f} {sp:>8.3f}{sig}")
print("\n* p < 0.10")

## 4. Scatter — Each Harvest Month vs. Production Change

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, (col, label) in zip(axes, cols_labels):
    ax.scatter(df[col], df["prod_chg"] * 100, alpha=0.65, s=30, edgecolors="none", color="steelblue")

    m, b, r, p_val, _ = stats.linregress(df[col], df["prod_chg"] * 100)
    xr_ = np.linspace(df[col].min(), df[col].max(), 200)
    ax.plot(xr_, m * xr_ + b, "--", color="tomato", lw=1.5,
            label=f"r²={r**2:.2f}  p={p_val:.3f}")

    ax.axhline(0, color="black", lw=0.5)
    ax.set_xlabel(label, fontsize=8)
    ax.set_ylabel("YoY Prod Change (%)" if ax is axes[0] else "")
    ax.legend(fontsize=7)

plt.suptitle("Harvest-Window Rain vs. YoY Hazelnut Production Change — Turkey", y=1.02)
plt.tight_layout()
plt.show()

## 5. Wet vs. Dry Harvest Years (Median Split on Aug–Oct Total)

In [ ]:
harvest_med = df["harvest_mm"].median()
df["wet_harvest"] = df["harvest_mm"] > harvest_med

wet = df[df["wet_harvest"]]
dry = df[~df["wet_harvest"]]

print(f"Aug–Oct median: {harvest_med:.0f} mm")
print(f"\nWet harvest years (n={len(wet)}):  mean prod chg = {wet['prod_chg'].mean():+.2%}  "
      f"median = {wet['prod_chg'].median():+.2%}")
print(f"Dry harvest years (n={len(dry)}):  mean prod chg = {dry['prod_chg'].mean():+.2%}  "
      f"median = {dry['prod_chg'].median():+.2%}")

t, p = stats.ttest_ind(wet["prod_chg"], dry["prod_chg"], equal_var=False)
print(f"\nWelch t-test: t={t:.2f}, p={p:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot
axes[0].boxplot(
    [dry["prod_chg"] * 100, wet["prod_chg"] * 100],
    labels=[f"Dry (<{harvest_med:.0f}mm)", f"Wet (≥{harvest_med:.0f}mm)"],
    patch_artist=True,
    boxprops=dict(facecolor="#90caf9"),
    medianprops=dict(color="tomato", lw=2),
    widths=0.5,
)
axes[0].axhline(0, color="black", lw=0.5, ls=":")
axes[0].set_ylabel("YoY Production Change (%)")
axes[0].set_title("Production Change by Harvest-Window Rain")

# Timeline
colors = df["wet_harvest"].map({True: "#ef9a9a", False: "#90caf9"})
axes[1].bar(df["year"], df["prod_chg"] * 100, color=colors, edgecolor="none", width=0.85)
axes[1].axhline(0, color="black", lw=0.6)
axes[1].set_xlabel("Year")
axes[1].set_ylabel("YoY Production Change (%)")
axes[1].set_title("Blue = dry harvest  |  Red = wet harvest")
axes[1].tick_params(axis="x", rotation=45, labelsize=7)
plt.tight_layout()
plt.show()

## 6. Month-by-Month Signal — Which Month Matters Most?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, (col, label) in zip(axes, cols_labels[:3]):
    med = df[col].median()
    wet_m = df[df[col] >  med]["prod_chg"] * 100
    dry_m = df[df[col] <= med]["prod_chg"] * 100
    r, p = stats.pearsonr(df[col], df["prod_chg"])

    ax.boxplot(
        [dry_m, wet_m],
        labels=[f"Dry\n(≤{med:.0f}mm)", f"Wet\n(>{med:.0f}mm)"],
        patch_artist=True,
        boxprops=dict(facecolor="#90caf9"),
        medianprops=dict(color="tomato", lw=2),
        widths=0.45,
    )
    ax.axhline(0, color="black", lw=0.4, ls=":")
    ax.set_title(f"{label.split(' ')[0]}\nr={r:+.3f}  p={p:.3f}", fontsize=9)
    ax.set_ylabel("YoY Prod Change (%)" if ax is axes[0] else "")

plt.suptitle("Production Change by Individual Harvest Month (median split)", y=1.03)
plt.tight_layout()
plt.show()

## 7. Extreme Wet Years — Top Decile

In [ ]:
q90 = df["harvest_mm"].quantile(0.90)
extreme = (
    df[df["harvest_mm"] >= q90]
    [["year", "harvest_mm", "aug_mm", "sep_mm", "oct_mm", "production_mt", "prod_chg"]]
    .copy()
)
extreme["prod_chg_pct"] = (extreme["prod_chg"] * 100).round(1)
extreme = extreme.drop(columns="prod_chg").sort_values("harvest_mm", ascending=False)

print(f"Top-decile threshold: {q90:.0f} mm (Aug–Oct combined)\n")
print(extreme.to_string(index=False))
print(f"\nMean prod change in top-decile wet years: {extreme['prod_chg_pct'].mean():+.1f}%")

## 8. Summary

| | August | September | October | Aug–Oct |
|---|---|---|---|---|
| Mean (mm) | ~86 | ~113 | ~137 | ~336 |
| Range (mm) | 24–236 | 29–232 | 24–262 | 174–614 |

**Interpretation guide:**
- Negative r = more rain → lower production (supports mold/rot thesis)
- Weak r despite the thesis: hazelnut interannual variance is dominated by frost events and the biennial on/off cycle — rain's marginal effect may be real but small at the annual aggregate level
- The signal may be sharper in **price** than volume (wet-harvest quality discount)
- A multivariate regression controlling for `frost_dh` + `is_on_year` would isolate the rain coefficient more cleanly